[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C67_LLM_Judge_Course/00_setup/00_environment_check.ipynb)

# 00 · 课程总览与环境（judge 模拟器 / 误差如何变成排名错误 / 三种用法的分辨力 / 体检清单）

目标：在读完讲解之后，建立一个**你知道真值**的 judge 实验台——
后面五个模块的所有去偏、校准、排名方法，都会在这个实验台上被验证。

本 notebook 你会亲手实现：
1. **环境自检**
2. **可控 judge 模拟器** —— 真实质量 + 五个显式偏差旋钮（位置 / 长度 / 自偏好 / 严厉度 / 噪声）
3. **judge 误差如何变成排名错误** —— 一致率 82% 的 judge，选错模型的概率是多少
4. **三种用法的分辨力对比** —— pointwise / pairwise / 校验，谁更能区分两个质量接近的模型
5. **judge 体检六项清单** —— 把「这个 judge 能不能用」变成一个可以打分的检查表

> 心智模型：**judge 不是裁判，是测量仪器。它的误差与被测物强相关，
> 因此不会在平均之后抵消，而是会系统性地奖励某一类输出。**

## 0 · 环境自检

In [ ]:
import sys, math, json, itertools
from collections import Counter, defaultdict

import numpy as np

print('Python :', sys.version.split()[0])
print('numpy  :', np.__version__)
rng = np.random.default_rng(0)
assert rng.integers(0, 10, size=3).shape == (3,)
print('\n✅ 环境就绪：本课全部内容 CPU 可跑、断网可跑，不需要任何 API key。')

## 1 · 可控 judge 模拟器

核心设计：每个回答有一个**真实质量** `q`（我们知道，judge 不知道）和一些**表面属性**
（长度、是不是 judge 自己写的）。judge 看到的是

$$\hat{q} = q + \underbrace{b_{\text{len}}\cdot z_{\text{len}} + b_{\text{self}}\cdot \mathbb{1}[\text{自己写的}] + b_{\text{pos}}\cdot \mathbb{1}[\text{排在后面}]}_{\text{系统偏差 }f} + \varepsilon$$

**关键点：$f$ 与内容质量无关，所以它不会被平均掉。**

In [ ]:
class JudgeSim:
    """可控的 LLM judge 模拟器。所有偏差都是显式旋钮，方便验证去偏方法。"""

    def __init__(self, b_len=0.0, b_self=0.0, b_pos=0.0, noise=0.3,
                 severity=0.0, seed=0):
        self.b_len = b_len          # 长度偏差：每 1 个标准差的长度，加多少分
        self.b_self = b_self        # 自偏好：judge 评自己的输出时加多少分
        self.b_pos = b_pos          # 位置偏差：排在第二位的答案加多少分
        self.noise = noise          # 随机噪声的标准差
        self.severity = severity    # 严厉度：整体平移（pointwise 打分时才可见）
        self.rng = np.random.default_rng(seed)

    def _perceived(self, q, z_len, is_self, is_second):
        return (q + self.b_len * z_len + self.b_self * is_self
                + self.b_pos * is_second - self.severity
                + self.rng.normal(0, self.noise, size=np.shape(q)))

    def pointwise(self, q, z_len=0.0, is_self=0, scale=5,
                  noise_mult=2.0, compression=0.6):
        """返回 1..scale 的整数分。真实质量 q 假定在 [0,1]。
        两个经验事实被显式建模进来：
          ① 绝对打分比相对比较更难，噪声更大（noise_mult）；
          ② judge 很少给 1 分和 5 分，分数向中间收缩（compression）。"""
        q = np.asarray(q, dtype=float)
        v = (q + self.b_len * z_len + self.b_self * is_self - self.severity
             + self.rng.normal(0, self.noise * noise_mult, size=np.shape(q)))
        v = 0.5 + compression * (v - 0.5)
        return np.clip(np.round(v * (scale - 1) + 1), 1, scale)

    def pairwise(self, qa, qb, z_len_a=0.0, z_len_b=0.0,
                 self_a=0, self_b=0, a_first=True):
        """返回 1 表示判 A 赢，0 表示判 B 赢。a_first=False 时 A 被放在第二位。"""
        va = self._perceived(np.asarray(qa, dtype=float), z_len_a, self_a, 0 if a_first else 1)
        vb = self._perceived(np.asarray(qb, dtype=float), z_len_b, self_b, 1 if a_first else 0)
        return (va > vb).astype(int)


rng = np.random.default_rng(1)
N = 4000
qa = rng.uniform(0, 1, N)
qb = rng.uniform(0, 1, N)
truth = (qa > qb).astype(int)                      # 真实的「谁更好」

clean = JudgeSim(noise=0.15, seed=2)
biased = JudgeSim(b_len=0.25, b_pos=0.15, noise=0.15, seed=3)

acc_clean = float((clean.pairwise(qa, qb) == truth).mean())
z_len_a = rng.normal(0, 1, N)                       # A 的长度（标准化后）
acc_biased = float((biased.pairwise(qa, qb, z_len_a=z_len_a) == truth).mean())
print(f'无偏 judge 与真值一致率: {acc_clean:.1%}')
print(f'有偏 judge 与真值一致率: {acc_biased:.1%}')
assert acc_clean > acc_biased
print('\n✅ 实验台就位。注意：有偏 judge 的一致率只掉了几个点——')
print('   但它掉分的方式是**系统性的**（偏向长答案、偏向后一个位置），这才是真正的问题。')

In [ ]:
# 系统偏差不会被平均掉：看「长答案」这个子群上的胜率
long_mask = z_len_a > 1.0
short_mask = z_len_a < -1.0
pred = biased.pairwise(qa, qb, z_len_a=z_len_a)

print(f"{'子群':<14}{'真实 A 胜率':>12}{'judge 判 A 胜率':>18}{'偏差':>10}")
for name, m in [('A 写得很长', long_mask), ('A 写得很短', short_mask), ('全体', np.ones(N, bool))]:
    t, p = truth[m].mean(), pred[m].mean()
    print(f'{name:<14}{t:>12.1%}{p:>18.1%}{p-t:>+10.1%}')

bias_long = pred[long_mask].mean() - truth[long_mask].mean()
bias_short = pred[short_mask].mean() - truth[short_mask].mean()
assert bias_long > 0.05 and bias_short < -0.05
print('\n✅ 全体上的偏差看起来不大，但拆开看：长答案被高估、短答案被低估。')
print('   **这就是「系统偏差不会被平均掉」的可见形式** —— 它只是在子群之间互相掩盖。')
print('   一旦你的模型开始写得更长，这个偏差就会全部变成虚假的分数提升。')

## 2 · judge 误差如何变成排名错误

真正要回答的问题不是「judge 准不准」，而是**「用这个 judge 做选型，选错的概率是多少」**。

In [ ]:
def selection_error_rate(judge, q_gap, n_items=200, n_trials=400, seed=0):
    """两个模型真实质量差 q_gap，用 judge 跑 n_items 道题的成对比较，
    以多数胜负决定选谁。返回选错的比例。"""
    rng = np.random.default_rng(seed)
    wrong = 0
    for _ in range(n_trials):
        base = rng.uniform(0, 1 - q_gap, n_items)
        qa_, qb_ = base + q_gap, base            # A 真实更好
        wins = judge.pairwise(qa_, qb_).mean()
        if wins <= 0.5:
            wrong += 1
    return wrong / n_trials

print(f"{'真实质量差':>12}{'干净 judge':>14}{'有偏 judge':>14}{'弱 judge':>12}")
weak = JudgeSim(noise=0.45, seed=7)
for gap in [0.02, 0.05, 0.10, 0.20]:
    e1 = selection_error_rate(JudgeSim(noise=0.15, seed=11), gap)
    e2 = selection_error_rate(JudgeSim(b_len=0.25, noise=0.15, seed=12), gap)
    e3 = selection_error_rate(weak, gap)
    print(f'{gap:>12.0%}{e1:>14.1%}{e2:>14.1%}{e3:>12.1%}')

e_small = selection_error_rate(JudgeSim(noise=0.45, seed=13), 0.02)
e_large = selection_error_rate(JudgeSim(noise=0.45, seed=13), 0.20)
assert e_small > e_large
print('\n✅ 质量差越小，judge 噪声越致命。这解释了一个常见现象：')
print('   judge 在区分「强模型 vs 弱模型」时很可靠，在区分「两个都很强的模型」时几乎在抛硬币——')
print('   而后者恰恰是我们最常需要它做的判断。')

## 3 · 三种用法的分辨力对比

同样的 judge 能力，pointwise / pairwise / 校验三种用法能区分多小的质量差？

In [ ]:
def pointwise_discrimination(judge, q_gap, n=3000, scale=5, seed=0):
    """pointwise：给两个模型各打 n 个分，看均分差的 t 统计量（分辨力）。"""
    rng = np.random.default_rng(seed)
    base = rng.uniform(0, 1 - q_gap, n)
    sa = judge.pointwise(base + q_gap, scale=scale)
    sb = judge.pointwise(base, scale=scale)
    d = sa - sb
    return float(d.mean() / (d.std(ddof=1) / math.sqrt(n) + 1e-12))

def pairwise_discrimination(judge, q_gap, n=3000, seed=0):
    rng = np.random.default_rng(seed)
    base = rng.uniform(0, 1 - q_gap, n)
    w = judge.pairwise(base + q_gap, base)
    p = w.mean()
    se = math.sqrt(max(p * (1 - p), 1e-12) / n)
    return float((p - 0.5) / se)

def verification_discrimination(q_gap, n=3000, base_pass=0.5, seed=0):
    """校验：条件可判定，judge 几乎不出错；而且天然是配对的
    （同一道题、同一个条件，两个模型各查一次），任务难度这个方差源被直接消掉。"""
    rng = np.random.default_rng(seed)
    difficulty = rng.uniform(0, 1, n)                 # 每道题的难度（两个模型共享）
    a = (difficulty < np.clip(base_pass + q_gap, 0, 1)).astype(float)
    b = (difficulty < base_pass).astype(float)
    d = a - b
    return float(d.mean() / (d.std(ddof=1) / math.sqrt(n) + 1e-12))

j = JudgeSim(noise=0.3, seed=5)
print(f"{'质量差':>8}{'pointwise(1-5)':>18}{'pairwise':>12}{'校验':>10}   (|t| > 2 才算能分辨)")
for gap in [0.02, 0.05, 0.10]:
    print(f'{gap:>8.0%}{pointwise_discrimination(j, gap):>18.2f}'
          f'{pairwise_discrimination(j, gap):>12.2f}{verification_discrimination(gap):>10.2f}')

t_point = pointwise_discrimination(j, 0.05)
t_pair = pairwise_discrimination(j, 0.05)
t_ver = verification_discrimination(0.05)
assert t_ver > t_pair > t_point
print(f'\n5% 质量差下的分辨力: 校验 {t_ver:.2f} > pairwise {t_pair:.2f} > pointwise {t_point:.2f}')
print('\n✅ 同样的样本量，pairwise 的分辨力明显高于 pointwise。')
print('   原因：1-5 的整数分把信息量化掉了一大半（05% 的质量差根本不足以让分数跳一档）。')
print('   而校验的分辨力最高——因为它把主观判断换成了可判定的条件。')
print('   → 选择顺序：能写成校验就写成校验，其次 pairwise，最后才 pointwise。')

## 4 · judge 体检六项清单

把「这个 judge 能不能用」变成一个可打分的检查表。六项全过才叫「体检过了」。

In [ ]:
JUDGE_CHECKS = [
    ('human_agreement', '与人类标注的一致率（必须同时给出人类之间的一致率作为上界）'),
    ('position_bias',   '交换 A/B 位置后判断翻转的比例（swap 一致性）'),
    ('length_bias',     '控制质量后，长度对胜率的边际影响'),
    ('self_preference', '评自己产出 vs 评他人产出的分差'),
    ('discrimination',  '能不能区分两个已知有 X 点差距的模型（分辨力）'),
    ('drift',           '同一批样本隔一段时间重判，结论是否一致（模型/prompt 漂移）'),
]

def judge_audit(report):
    passed = [k for k, _ in JUDGE_CHECKS if report.get(k) is True]
    missing = [d for k, d in JUDGE_CHECKS if not report.get(k)]
    return len(passed) / len(JUDGE_CHECKS), missing

typical_paper = {'human_agreement': True, 'position_bias': True,
                 'length_bias': False, 'self_preference': False,
                 'discrimination': False, 'drift': False}
score, missing = judge_audit(typical_paper)
print(f'一份「典型技术报告」的 judge 体检得分: {score:.0%}\n')
for m in missing:
    print('  ✗', m)
assert abs(score - 2 / 6) < 1e-9
print('\n✅ 这四项缺口恰好是本课 02-03 模块的主题。')
print('   注意 human_agreement 打勾还不够——**必须同时给出人类之间的一致率**，')
print('   否则你无法判断 82% 是「接近天花板」还是「还差得远」。')

## ✏️ 练习 1：把一致率翻译成「选错的概率」

实现 `flip_probability(agreement, n_items)`：judge 在单题上与真值一致率为 `agreement`，
用 `n_items` 道题的多数投票做选型，返回**多数投票判错**的概率
（二项分布尾部，`n_items` 取偶数时平局按判错的一半算）。

In [ ]:
def flip_probability(agreement, n_items):
    # TODO：用 math.comb 直接求和，不要用蒙特卡洛
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
assert abs(flip_probability(1.0, 100)) < 1e-12
assert abs(flip_probability(0.5, 100) - 0.5) < 1e-9      # 完全随机 → 一半概率判错
p60 = flip_probability(0.60, 100)
p82 = flip_probability(0.82, 100)
assert p82 < p60
for n_ in [10, 50, 100, 400]:
    print(f'n={n_:>4}  一致率60% → 判错 {flip_probability(0.60, n_):.2%} | '
          f'一致率82% → 判错 {flip_probability(0.82, n_):.4%}')
print('✅ 练习 1 通过：一致率 60% 的 judge 在 100 道题上就已经很可靠了——')
print('   **前提是它的误差是随机的**。系统偏差不会随 n 增大而消失（第 1 节），')
print('   所以这个公式只适用于随机误差部分，这正是要先量偏差再算样本量的原因。')

## ✏️ 练习 2：分离随机误差与系统偏差

实现 `decompose_error(pred, truth, group_mask)`：返回
`(总错误率, 组内偏差, 组外偏差)`，其中偏差定义为 `pred.mean() - truth.mean()`。
用它验证：总错误率相同的两个 judge，可能一个是随机误差、一个是系统偏差。

In [ ]:
def decompose_error(pred, truth, group_mask):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
rng = np.random.default_rng(21)
n = 4000
truth_v = rng.integers(0, 2, n)
mask = rng.random(n) < 0.5
# judge R：纯随机误差，两个子群偏差都接近 0
pred_R = np.where(rng.random(n) < 0.075, 1 - truth_v, truth_v)
# judge S：系统偏差，只在 mask 组里偏向判 1
pred_S = np.where(mask & (rng.random(n) < 0.30), 1, truth_v)

for name, pred in [('R 随机误差', pred_R), ('S 系统偏差', pred_S)]:
    err, bin_, bout = decompose_error(pred, truth_v, mask)
    print(f'{name:<12} 错误率 {err:.1%} | 组内偏差 {bin_:+.3f} | 组外偏差 {bout:+.3f}')

err_R, bin_R, bout_R = decompose_error(pred_R, truth_v, mask)
err_S, bin_S, bout_S = decompose_error(pred_S, truth_v, mask)
assert abs(bin_R) < 0.05 and abs(bout_R) < 0.05
assert bin_S > 0.08 and abs(bout_S) < 0.02
assert abs(err_R - err_S) < 0.03, '两个 judge 的总错误率应当接近'
print('\n✅ 练习 2 通过：只看错误率，两个 judge 差不多；')
print('   拆到子群看，S 在一个子群上系统性偏向判 1——多跑一万题也消不掉。')
print('   **这就是为什么 judge 体检必须按子群做，而不是只报一个总一致率。**')

## ✏️ 练习 3：分辨力所需的样本量

实现 `n_for_discrimination(win_rate, target_z=2.0)`：judge 判 A 胜的概率是 `win_rate`
（真值应为 0.5 才叫无差异），求要让 $z = (p-0.5)/\sqrt{p(1-p)/n}$ 达到 `target_z`
所需的最小样本量 $n$（向上取整）。

In [ ]:
def n_for_discrimination(win_rate, target_z=2.0):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
n55 = n_for_discrimination(0.55)
n52 = n_for_discrimination(0.52)
assert n52 > n55
p = 0.55
z = (p - 0.5) / math.sqrt(p * (1 - p) / n55)
assert z >= 2.0 - 1e-9
for w in [0.75, 0.65, 0.55, 0.52, 0.51]:
    print(f'judge 判 A 胜率 {w:.0%} → 要达到 z=2 需要 {n_for_discrimination(w):>6,} 道题')
print('✅ 练习 3 通过：胜率 51% 需要上万道题——')
print('   而「两个都很强的模型」的真实胜率差往往就在 51%-53% 这个区间。')

## ✏️ 练习 4：judge 体检的加权得分

实现 `weighted_judge_audit(report, weights)`：按权重算体检完整度。
用它验证「缺 human_agreement 比缺 drift 掉分更多」。

In [ ]:
def weighted_judge_audit(report, weights):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
W = {'human_agreement': 3, 'position_bias': 2, 'length_bias': 2,
     'self_preference': 1, 'discrimination': 2, 'drift': 1}
all_true = {k: True for k, _ in JUDGE_CHECKS}
assert abs(weighted_judge_audit(all_true, W) - 1.0) < 1e-12
assert abs(weighted_judge_audit({k: False for k, _ in JUDGE_CHECKS}, W)) < 1e-12
no_human = dict(all_true, human_agreement=False)
no_drift = dict(all_true, drift=False)
assert weighted_judge_audit(no_human, W) < weighted_judge_audit(no_drift, W)
print(f'只缺 human_agreement: {weighted_judge_audit(no_human, W):.1%}')
print(f'只缺 drift:          {weighted_judge_audit(no_drift, W):.1%}')
print(f'典型技术报告:        {weighted_judge_audit(typical_paper, W):.1%}')
print('✅ 练习 4 通过：六项不等权——没有与人类的对照，其余五项做得再好也说明不了 judge 是对的。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def flip_probability(agreement, n_items):
    p = float(agreement)
    half = n_items / 2
    total = 0.0
    for k in range(n_items + 1):
        prob = math.comb(n_items, k) * (p ** k) * ((1 - p) ** (n_items - k))
        if k < half:
            total += prob                 # 少数正确 → 判错
        elif k == half:
            total += 0.5 * prob           # 平局 → 一半概率判错
    return total

In [ ]:
# 练习 2 参考答案
def decompose_error(pred, truth, group_mask):
    pred = np.asarray(pred, dtype=float)
    truth = np.asarray(truth, dtype=float)
    m = np.asarray(group_mask, dtype=bool)
    err = float((pred != truth).mean())
    bias_in = float(pred[m].mean() - truth[m].mean()) if m.any() else 0.0
    bias_out = float(pred[~m].mean() - truth[~m].mean()) if (~m).any() else 0.0
    return (err, bias_in, bias_out)

In [ ]:
# 练习 3 参考答案
def n_for_discrimination(win_rate, target_z=2.0):
    p = float(win_rate)
    if abs(p - 0.5) < 1e-12:
        return float('inf')
    return math.ceil(target_z ** 2 * p * (1 - p) / (p - 0.5) ** 2)

In [ ]:
# 练习 4 参考答案
def weighted_judge_audit(report, weights):
    total = sum(weights.values())
    got = sum(w for k, w in weights.items() if report.get(k))
    return got / total if total else 0.0

---
## 🧪 真实工程胶囊：一个可以直接用的 judge 调用骨架

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════
# A. pairwise judge 的最小可用 prompt（结构化输出，便于解析与统计）
# ══════════════════════════════════════════════════════════════════
JUDGE_PROMPT = "
".join([
    "You are evaluating two responses to the same user request.",
    "",
    "<request>{request}</request>",
    "<response_a>{a}</response_a>",
    "<response_b>{b}</response_b>",
    "",
    "Evaluate against these criteria, in this order of importance:",
    "1. Correctness: are the factual claims accurate?",
    "2. Instruction following: does it do what was asked?",
    "3. Completeness: are required parts missing?",
    "4. Clarity.",
    "",
    "IMPORTANT: Do NOT reward length. A shorter response that fully answers",
    "is better than a longer one that padded. Do not reward confident tone.",
    "",
    "Return JSON only:",
    '{{"reasoning": "<2-3 sentences>", "verdict": "A" | "B" | "tie"}}',
])

# ══════════════════════════════════════════════════════════════════
# B. 必须成对调用：swap 一致性（02 模块的位置偏差探针）
# ══════════════════════════════════════════════════════════════════
def judge_pair(client, request, a, b):
    v1 = call_judge(client, JUDGE_PROMPT.format(request=request, a=a, b=b))
    v2 = call_judge(client, JUDGE_PROMPT.format(request=request, a=b, b=a))  # 交换
    # v2 的 "A" 指的是原来的 b —— 解析时必须翻译回来
    v2_translated = {"A": "B", "B": "A", "tie": "tie"}[v2["verdict"]]
    consistent = (v1["verdict"] == v2_translated)
    return {"verdict_1": v1["verdict"], "verdict_2": v2_translated,
            "consistent": consistent,
            # 不一致时记为 tie，是最保守也最常见的处理方式
            "final": v1["verdict"] if consistent else "tie"}
# 报告规范：swap 一致率必须与胜率一起报。一致率 < 80% 时，胜率基本不可信。

# ══════════════════════════════════════════════════════════════════
# C. 每次 judge 运行都要记的字段（否则事后无法做偏差分析）
# ══════════════════════════════════════════════════════════════════
JUDGE_ROW = {
  "item_id": "...", "judge_model": "...", "judge_prompt_hash": "e3a1…",
  "a_model": "...", "b_model": "...",
  "a_len_tokens": 412, "b_len_tokens": 890,     # ← 长度偏差分析必需
  "order": "ab",                                 # ← 位置偏差分析必需
  "verdict": "A", "verdict_swapped": "A", "consistent": True,
  "reasoning": "...", "latency_ms": 1830, "cost_usd": 0.0031,
}
# 缺 a_len_tokens / order 这两个字段，02 模块的所有去偏方法都做不了。
'''
print(RECIPE)

### 小结

| 你学到的 | 一句话 | 展开在 |
|---|---|---|
| judge 是测量仪器 | 它的误差与被测物强相关，不会被平均掉 | 全课 |
| 三种用法 | 能写成校验就写成校验，其次 pairwise，最后 pointwise | 01 / 05 |
| 四个结构性难点 | 金标准有噪声 / 系统偏差 / 被评对象会适应 / 成对到排名 | 02–05 |
| 误差→排名错误 | 质量差越小，judge 噪声越致命；51% 的胜率差需要上万道题 | 04 |
| 体检六项 | 一致率必须配着人类上界一起报 | 03 |

下一模块：**01 · Judge 设计**——打分粒度、rubric 拆解、参考答案、CoT 与结构化输出，
以及「为什么 1-5 分制会把你的判别力砍掉一半」。